In [1]:
from pyr_suite2p_based import load_cells_with_spike_pkls

# Option 1: DB
cells = load_cells_with_spike_pkls(db_path=r"Z:\...\PyrLow.csv")

# Option 2: direct paths
cells = load_cells_with_spike_pkls(cell_paths=[r"Z:\...\cell12", r"Z:\...\cell37"])


FileNotFoundError: [Errno 2] No such file or directory: 'Z:\\...\\PyrLow.csv'

In [1]:
from pyr_suite2p_based import load_cells_with_spike_pkls
cells = load_cells_with_spike_pkls(db_path=r"Z:\...\PyrLow.csv")


FileNotFoundError: [Errno 2] No such file or directory: 'Z:\\...\\PyrLow.csv'

In [3]:
from pyr_suite2p_based import (
    load_cells_with_spike_pkls,
    evaluate_suite2p_spks_as_continuous_activity,
)

# load cells (DB mode or direct paths)
cells = load_cells_with_spike_pkls(
    db_path=r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\PyrLowFR.csv"
    # or: cell_paths=[r"Z:\...\cell1", r"Z:\...\cell2"]
)

res = evaluate_suite2p_spks_as_continuous_activity(
    loaded_cells=cells,
    out_subdir="suite2p_spks_eval_continuous",
    smooth_sigma_s=0.2,
    max_lag_s=0.5,
    z_thresh=2.0,
    save_svg=True,  # True if kaleido is available
)
import os
import numpy as np

valid = res.loc[res["ok"] == True].copy().reset_index(drop=True)

mapping = valid[["cell_path", "state"]].copy()
mapping["subplot_row"] = np.arange(1, len(mapping) + 1)  # 1-based row index
mapping["subplot_title"] = mapping["cell_path"].map(lambda p: os.path.basename(str(p))) + " | " + mapping["state"]

display(mapping)

# display(res.head(20))
# print("valid rows:", int(res["ok"].fillna(False).sum()), " / ", len(res))
# display(res.head(20))
# print("valid rows:", int(res["ok"].fillna(False).sum()), " / ", len(res))

# save summary table
out_csv = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\suite2p_spks_eval_summary.csv"
res.to_csv(out_csv, index=False)
print(out_csv)


,cell_path,state,subplot_row,subplot_title
0,Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\...,main,1,cell0 | main
1,Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\...,main,2,cell1 | main
2,Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\L\...,main,3,cell0 | main
3,Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc41\l\...,main,4,cell0 | main
4,Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\Wh...,main,5,cell0 | main
5,Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc41\RW...,m0,6,cell1 | m0
6,Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc41\RW...,r0,7,cell1 | r0
7,Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc41\RW...,m1,8,cell1 | m1
8,Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc41\RW...,r1,9,cell1 | r1
9,Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\RL...,main,10,cell4 | main


Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\suite2p_spks_eval_summary.csv


In [2]:
import importlib
import pyr_suite2p_based as psb
importlib.reload(psb)

from pyr_suite2p_based import (
    load_cells_with_spike_pkls,
    evaluate_suite2p_spks_as_continuous_activity,
)
cells = load_cells_with_spike_pkls(
    db_path=r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\PyrLowFR.csv"
    # or: cell_paths=[r"Z:\...\cell1", r"Z:\...\cell2"]
)

res = evaluate_suite2p_spks_as_continuous_activity(
    loaded_cells=cells,
    out_subdir="suite2p_spks_eval_continuous",
    smooth_sigma_s=0.2,
    max_lag_s=0.5,
    z_thresh=2.0,
    error_z_mode="fr_bin",   # new mode
    save_svg=True,
)


c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\kaleido\scopes\base.py:185: DeprecationWarning:

setDaemon() is deprecated, set the daemon attribute instead



In [4]:
import pandas as pd
import numpy as np
import os

# Use pooled event table generated by the analysis
evt_csv = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\Pyr\suite2p_vs_realfr\countineous_aproch\all_cells_event_under_over_by_type_size.csv"
evt = pd.read_csv(evt_csv)

# Keep only complex-underestimation events
sub = evt[
    (evt["category"].astype(str).str.lower() == "under") &
    (evt["event_class"].astype(str).str.lower() == "complex")
].copy()

# Rank by number of complex underestimated events
ranked = (
    sub.groupby(["cell_path", "cell_name"], as_index=False)
       .size()
       .rename(columns={"size": "n_complex_underestimated"})
       .sort_values("n_complex_underestimated", ascending=False)
       .reset_index(drop=True)
)

# 1-based rank
ranked["rank"] = np.arange(1, len(ranked) + 1)

# Optional: add total complex events and fraction underestimated
total_complex = (
    evt[evt["event_class"].astype(str).str.lower() == "complex"]
    .groupby(["cell_path", "cell_name"], as_index=False)
    .size()
    .rename(columns={"size": "n_complex_total"})
)

ranked = ranked.merge(total_complex, on=["cell_path", "cell_name"], how="left")
ranked["frac_complex_underestimated"] = ranked["n_complex_underestimated"] / ranked["n_complex_total"]

# Show ranked list
display(
    ranked[["rank", "cell_name", "cell_path", "n_complex_underestimated", "n_complex_total", "frac_complex_underestimated"]]
)

# Optional: save
out_csv = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\Pyr\suite2p_vs_realfr\countineous_aproch\ranked_cells_complex_underestimation.csv"
ranked.to_csv(out_csv, index=False)
print(out_csv)


,rank,cell_name,cell_path,n_complex_underestimated,n_complex_total,frac_complex_underestimated
0,1,cell1,Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC44\L\...,16,21,0.761905
1,2,cell1,Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\Wh...,14,14,1.000000
2,3,cell2,Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\RL...,12,16,0.750000
3,4,cell0,Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc46\RL...,12,12,1.000000
4,5,cell6,Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC44\L\...,12,19,0.631579
5,6,cell4,Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC44\L\...,11,12,0.916667
6,7,cell0,Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\Wh...,11,12,0.916667
7,8,cell1,Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\...,10,13,0.769231
8,9,cell0,Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC40\R\...,9,9,1.000000
9,10,cell1,Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc41\RW...,8,9,0.888889


Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\Pyr\suite2p_vs_realfr\countineous_aproch\ranked_cells_complex_underestimation.csv


In [3]:
# 1) Negative lag-corrected correlation (direct spks-vs-FR after lag search)
neg_lag = (
    res[(res["ok"] == True) & (res["best_lag_pearson_r"] < 0)]
    [["cell_path", "state", "best_lag_pearson_r", "best_lag_s"]]
    .sort_values("best_lag_pearson_r")
)

display(neg_lag)
print("count:", len(neg_lag))


,cell_path,state,best_lag_pearson_r,best_lag_s
8,Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc41\RW...,r1,-0.287066,0.466667


count: 1


In [1]:
from pyr_suite2p_based import (
    load_cells_with_spike_pkls,
    evaluate_suite2p_spks_as_continuous_activity,
)

# load cells (DB mode or direct paths)
cells = load_cells_with_spike_pkls(
    db_path=r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\PyrLowFR.csv"
    # or: cell_paths=[r"Z:\...\cell1", r"Z:\...\cell2"]
)
res = evaluate_suite2p_spks_as_continuous_activity(
    loaded_cells=cells,
    out_subdir="suite2p_spks_eval_continuous",
    smooth_sigma_s=0.2,
    max_lag_s=0.5,
    z_thresh=2.0,
    error_z_mode="shift_null",
    shift_null_window_sizes_s=(0.1,0.25, 0.5, 1.0),
    shift_null_n_shifts=200,
    shift_null_min_supported_scales=2,
    shift_null_min_valid_frac=0.80,
    shift_null_merge_gap_s=0.0,
    shift_null_seed=0,
    corrlation_thr=0.4,
    save_svg=True,
)


c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\kaleido\scopes\base.py:185: DeprecationWarning:

setDaemon() is deprecated, set the daemon attribute instead



In [2]:
import importlib, pyr_suite2p_based as psb
importlib.reload(psb)

cells = psb.load_cells_with_spike_pkls(
    db_path=r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\PyrLowFR.csv"
)

res_cas = psb.evaluate_cascade_as_continuous_activity(
    loaded_cells=cells,
    cascade_model_name="GC8_EXC_30Hz_smoothing50ms_high_noise",
    cascade_model_folder=r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\code\cal_vol_soma\Pretrained_models",
    smooth_sigma_s=0.2,
    max_lag_s=0.5,
    z_thresh=2.0,
    error_z_mode="standard",   # or "standard" / "fr_bin"
    shift_null_window_sizes_s=(0.1, 0.25, 0.5, 1.0),
    shift_null_n_shifts=200,
    corrlation_thr=0.3,
    #cascade_model_name="GC8_EXC_30Hz_smoothing50ms_high_noise",
    save_svg=True,
)


Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
Spike rate inference done.
S